In machine translation, there is rarely a single correct output. Two human translators may express the same meaning using different words or sentence structures. Evaluating every machine-generated translation by a human is also expensive.

Papineni et al. introduced the Bilingual Evaluation Understudy (BLEU) metric [@Papineni2002-wc] as an inexpensive, reproducible way to compare machine translations with human reference translations. BLEU measures lexical overlap. It does not measure meaning directly. Its original purpose was to compare translation systems over a corpus.

This section builds the metric by answering three questions. How do we prevent repeated words from inflating precision? How do we reward appropriate word order? How do we discourage translations that omit content?

## 1. Clipped n-gram counts

Consider the example from the original BLEU paper [@Papineni2002-wc].

| | Translation |
|---|---|
| Candidate | the the the the the the the |
| Reference 1 | the cat is on the mat |
| Reference 2 | there is a cat on the mat |

If you count a candidate word as correct whenever it occurs anywhere in a reference, all seven candidate words match. The resulting precision is $7/7=1$, despite the translation conveying almost no information.

The problem is that the same reference evidence is reused from the references. The first reference contains two occurrences of `the`, and the second contains one. BLEU therefore credits at most two occurrences, not seven and not three. Counts from alternative references are not added together.

Let $C(g)$ be the number of occurrences of an n-gram $g$ in the candidate, and $R_j(g)$ its count in reference $j$. The clipped count is

$$
C_{\mathrm{clip}}(g)=\min\left(C(g),\max_j R_j(g)\right).
$$

The modified precision at order $n$ is

$$
p_n=\frac{\sum_{g\in G_n} C_{\mathrm{clip}}(g)}{\sum_{g\in G_n} C(g)},
$$

where $G_n$ is the set of distinct candidate n-grams. In the repeated-word example, $p_1=2/7$. The denominator counts all candidate occurrences; the numerator counts only those supported by reference evidence.

This resembles precision in classification, because it measures how many proposed occurrences are supported. There is no conventional true-negative count, however, and BLEU does not explicitly compute recall.

In [ ]:
from collections import Counter
from math import exp, log

def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def modified_counts(candidate, references, n):
    counts = ngrams(candidate, n)
    ref_counts = [ngrams(ref, n) for ref in references]
    clipped = {g: min(v, max(r[g] for r in ref_counts))
               for g, v in counts.items()}
    return sum(clipped.values()), sum(counts.values())

refs = ["the cat is on the mat".split(),
        "there is a cat on the mat".split()]
candidate = ["the"] * 7
matches, total = modified_counts(candidate, refs, 1)
print(f"Modified unigram precision: {matches}/{total} = {matches/total:.3f}")

## 2. From words to sequences of words

Unigram precision cannot distinguish sentences containing the same words in different orders. Consider a single reference and two candidates.

| | Translation |
|---|---|
| Reference | the dog chased the cat |
| Candidate A | the dog chased the cat |
| Candidate B | the cat chased the dog |

Both candidates have perfect unigram precision. Candidate B, however, reverses the roles of the animals.

For candidate B, the bigrams are `the cat`, `cat chased`, `chased the`, and `the dog`. Three of these four occur in the reference. Its modified bigram precision is therefore $3/4$. Neither of its two four-grams occurs in the reference.

Higher-order n-grams provide evidence about local word order and phrasing. They do not, by themselves, establish grammatical correctness or semantic equivalence.

In [ ]:
reference = "the dog chased the cat".split()
candidates = {"Exact": reference, "Reversed roles": "the cat chased the dog".split()}
for name, tokens in candidates.items():
    print(name)
    for order in range(1, 5):
        a, b = modified_counts(tokens, [reference], order)
        print(f"  n={order}: {a}/{b} = {a/b:.3f}")

## 3. The brevity penalty

Suppose the reference is `the dog chased the cat across the garden` and the candidate is only `the dog chased the`. Every candidate n-gram of orders one through four is supported by the reference. All four modified precisions equal one, even though half the translation is missing.

BLEU addresses this failure with a brevity penalty. Let $c$ be the candidate length and $r$ the effective reference length. Then

$$
\operatorname{BP}(c,r)=
\begin{cases}
1, & c\ge r,\quad c>0,\\
\exp(1-r/c), & 0<c<r,\\
0, & c=0.
\end{cases}
$$

For the example, $c=4$, $r=8$, and the penalty is $e^{-1}\approx 0.368$. When multiple references exist, BLEU chooses the reference length closest to the candidate length, breaking ties in favor of the shorter reference. At corpus level, these selected lengths are summed.

The brevity penalty is not recall: it penalizes insufficient length but does not check which information was omitted. Overly long candidates receive no separate length penalty; unsupported extra n-grams typically lower modified precision.

In [ ]:
from math import exp


def brevity_penalty(c, r):
    return exp(min(0, 1 - r / c)) if c else 0.0


In [ ]:
import matplotlib.pyplot as plt

ratios = [i/100 for i in range(1, 201)]
plt.figure(figsize=(7, 3.5))
plt.plot(ratios, [brevity_penalty(x, 1) for x in ratios])
plt.axvline(1, linestyle="--", alpha=.5)
plt.scatter([.5], [exp(-1)], zorder=3, label="Half-length candidate")
plt.xlabel("Candidate / reference length")
plt.ylabel("Brevity penalty")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=.2)
plt.show()

## 4. Combining the components: BLEU-4

You now have two complementary mechanisms: modified precision at several n-gram orders and a penalty for short translations. BLEU combines them as

$$
\operatorname{BLEU}=\operatorname{BP}(c,r)\exp\left(\sum_{n=1}^{N}w_n\log p_n\right)
=\operatorname{BP}(c,r)\prod_{n=1}^{N}p_n^{w_n},
$$

where the weights are nonnegative and sum to one. Terms with zero weight are omitted. Standard BLEU-4 uses $N=4$ and $w_n=1/4$.

The geometric mean rewards translations that perform well across all selected orders. For example, if the four precisions are $(1,0.8,0.4,0.1)$, their arithmetic mean is $0.575$, whereas their geometric mean is approximately $0.423$. Poor higher-order agreement has a stronger effect on the geometric mean. If any positively weighted precision is zero, unsmoothed BLEU is zero.

This is a design choice, not a mathematical necessity. It is useful for corpus evaluation but can be excessively harsh for individual sentences.

## 5. From sentence-level examples to corpus-level BLEU

The original BLEU metric aggregates clipped counts across sentences before calculating precision. For candidate sentence $s$ with its own references, let $a_{s,n}$ and $b_{s,n}$ be its clipped and total n-gram counts. Then

$$
p_n^{\mathrm{corpus}}=\frac{\sum_s a_{s,n}}{\sum_s b_{s,n}}.
$$

The brevity penalty uses the total candidate length and the sum of effective reference lengths. Consequently, corpus BLEU is not the arithmetic mean of sentence BLEU scores. Longer sentences generally contribute more n-gram occurrences.

The following implementation exposes the calculation. It uses fixed-order, unsmoothed BLEU, defining it as zero if a required order has no matching occurrences (including when the corpus has no n-grams of that order).

In [ ]:
def corpus_bleu_from_counts(candidates, references, weights=(.25,)*4):
    assert len(candidates) == len(references) and all(references)
    c = sum(map(len, candidates))
    r = sum(min((len(ref) for ref in rs), key=lambda x: (abs(x-len(h)), x))
            for h, rs in zip(candidates, references))
    precisions = []
    for order in range(1, len(weights)+1):
        counts = [modified_counts(h, rs, order) for h, rs in zip(candidates, references)]
        a, b = map(sum, zip(*counts))
        precisions.append(a/b if b else 0.0)
    bp = brevity_penalty(c, r)
    active = [(w, p) for w, p in zip(weights, precisions) if w > 0]
    score = bp * exp(sum(w*log(p) for w,p in active)) if all(p>0 for w,p in active) else 0.0
    return score, precisions, bp

from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
refs = [["the dog chased the cat".split()],
        ["a small bird flew over the garden".split()]]
hyps = ["the cat chased the dog".split(), refs[1][0]]
score, precisions, bp = corpus_bleu_from_counts(hyps, refs)
print("Corpus precisions:", precisions)
print("Brevity penalty:", bp)
print(f"Our corpus BLEU: {score:.4f}")
print(f"NLTK corpus BLEU: {corpus_bleu(refs, hyps):.4f}")
assert abs(score-corpus_bleu(refs, hyps)) < 1e-12
sentence_mean = sum(corpus_bleu_from_counts([h], [rs])[0] for h,rs in zip(hyps,refs))/len(hyps)
print(f"Mean unsmoothed sentence BLEU: {sentence_mean:.4f}")

## 6. Investigating translation errors

The next experiment uses one reference and several candidate translations. Inspect the individual precisions before looking at the combined score. The truncation has excellent precision but incurs a length penalty. The paraphrase may preserve meaning yet have little lexical overlap. The negation changes the meaning while retaining many reference n-grams.

In [ ]:
ref = "the dog chased the cat across the garden".split()
examples = {
    "Exact match": "the dog chased the cat across the garden",
    "Reversed roles": "the cat chased the dog across the garden",
    "Truncated": "the dog chased the",
    "Negation": "the dog never chased the cat across the garden",
    "Paraphrase": "a canine pursued a feline through the yard",
}
labels, rows = [], []
print(f"{'Candidate':18} {'p1':>6} {'p2':>6} {'p3':>6} {'p4':>6} {'BP':>6} {'BLEU':>6}")
for name, sentence in examples.items():
    score, ps, bp = corpus_bleu_from_counts([sentence.split()], [[ref]])
    labels.append(name)
    rows.append(ps+[bp, score])
    print(f"{name:18}" + "".join(f" {x:6.3f}" for x in rows[-1]))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(rows, vmin=0, vmax=1, cmap="Blues", aspect="auto")
ax.set_xticks(range(6), ["p1", "p2", "p3", "p4", "BP", "BLEU"])
ax.set_yticks(range(len(labels)), labels)
for i, row in enumerate(rows):
    for j, val in enumerate(row):
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color="white" if val>.65 else "black")
fig.colorbar(im, ax=ax, label="Score")
fig.tight_layout()
plt.show()

## 7. Sentence-level BLEU and smoothing

Short sentences may have no matching four-grams, even when much of the translation is correct. Smoothing modifies the precision calculation to avoid a zero geometric mean. Different smoothing methods produce different scores; they should not be treated as interchangeable with original unsmoothed corpus BLEU.

NLTK exposes several smoothing methods described by Chen and Cherry [@Chen2014-uy]. The code below uses method 1, which adds a small constant to the numerator for orders with zero matches (NLTK also guards against zero denominators).

In [ ]:
ref = "the dog chased the cat".split()
hyp = "the cat chased the dog".split()
print("Unsmoothed (explicit zero):", corpus_bleu_from_counts([hyp], [[ref]])[0])
print("Smoothed NLTK:", sentence_bleu([ref], hyp, smoothing_function=SmoothingFunction().method1))

## 8. What BLEU measures in practice

BLEU is useful when comparing machine translation systems on the same test corpus under the same evaluation protocol. Its limitations become particularly visible when evaluating individual outputs. A faithful paraphrase can score poorly, while a translation with a critical factual error can score relatively well.

Tokenization, case handling, reference selection, smoothing and effective-order conventions can affect reported results. BLEU is often reported on a scale from 0 to 100, which is the 0 to 1 score multiplied by 100. For reproducible machine translation benchmarks, SacreBLEU provides standardized evaluation and a configuration signature [@Post2018-fm]. Semantic metrics such as COMET [@Rei2020-dm] add complementary evidence, but they do not guarantee correctness.
